# Business Understanding

## Project Overview
DukaSense is a proposed web-based business analytics tool for small-shop owners, kiosk operators and informal retailers. It is designed to turn structured sales and product records into clear business metrics, product insights and understandable recommendations. The project follows the hackathon theme “People First, AI Second” by prioritising transparency, usability and human judgement over AI for its own sake.

## Project Statement
Many small-shop owners and informal retailers work with handwritten notebooks, receipts or basic spreadsheets to track sales and stock. These records can contain useful information, but interpreting them consistently may take time and basic financial-analysis skills. Owners may find it difficult to quickly identify which products contribute most to sales, estimate gross profit, spot slow-moving items or recognise when stock is running low. DukaSense proposes to make these records easier to interpret through a simple dashboard and transparent, rules-based insights. 

**Problem focus**: How might we help small-shop owners turn everyday sales and stock records into clear, trustworthy information that supports better-informed business decisions?

# Data understanding

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [7]:
df= pd.read_csv("DukaSense_Combined_Synthetic_Dataset_3.csv")
df.head()

C:\Users\Shaun Raymond\AppData\Local\Temp\ipykernel_13080\1670666522.py:1: DtypeWarning: Columns (2: transaction_id, 3: sale_date, 4: sale_month, 6: shop_name, 7: shop_type, 8: county, 9: town, 10: neighbourhood, 11: opening_date, 12: shop_currency, 15: product_category, 16: product_unit, 26: inventory_snapshot_date, 29: stock_status, 37: record_origin, 46: snapshot_date, 49: last_sale_date, 52: period_start, 53: period_end, 54: threshold_note) have mixed types. Specify dtype option on import or set low_memory=False.
  df= pd.read_csv("DukaSense_Combined_Synthetic_Dataset_3.csv")


,record_type,sale_line_id,transaction_id,sale_date,sale_month,shop_id,shop_name,shop_type,county,town,...,balance_after_movement_units,snapshot_date,quantity_on_hand,units_sold_last_90_days,last_sale_date,slow_moving,threshold_units,period_start,period_end,threshold_note
0,SALE,SL00000281,TX-SH001-20250101-001,2025-01-01 09:43:55,2025-01,SH001,Tumaini General Store,duka,Nairobi,Nairobi,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SALE,SL00000574,TX-SH001-20250101-001,2025-01-01 14:29:40,2025-01,SH001,Tumaini General Store,duka,Nairobi,Nairobi,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,SALE,SL00000474,TX-SH001-20250101-001,2025-01-01 16:43:02,2025-01,SH001,Tumaini General Store,duka,Nairobi,Nairobi,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,SALE,SL00000671,TX-SH001-20250102-001,2025-01-02 15:20:55,2025-01,SH001,Tumaini General Store,duka,Nairobi,Nairobi,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,SALE,SL00000475,TX-SH001-20250102-001,2025-01-02 16:15:42,2025-01,SH001,Tumaini General Store,duka,Nairobi,Nairobi,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
df.shape

(28683, 55)

- `28,683` rows and `53` columns

In [13]:
for record_type, group in df.groupby("record_type"):
    missing_pct = group.isna().mean().mul(100)
    missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)

    print(f"\n{record_type}: {len(group):,} rows")
    display(missing_pct.round(1).to_frame("missing (%)"))


INVENTORY_SNAPSHOT: 132 rows


,missing (%)
sale_line_id,100.0
transaction_id,100.0
sale_date,100.0
sale_month,100.0
shop_name,100.0
shop_type,100.0
county,100.0
town,100.0
neighbourhood,100.0
opening_date,100.0



PRODUCT_CATALOGUE: 132 rows


,missing (%)
sale_line_id,100.0
transaction_id,100.0
sale_date,100.0
sale_month,100.0
shop_name,100.0
shop_type,100.0
county,100.0
town,100.0
neighbourhood,100.0
opening_date,100.0



SALE: 13,450 rows


,missing (%)
inventory_snapshot_date,100.0
current_quantity_on_hand,100.0
inventory_reorder_level,100.0
stock_status,100.0
opening_stock_units,100.0
total_purchased_units,100.0
total_sale_stock_out_units,100.0
total_stock_adjustment_units,100.0
ending_stock_units_from_ledger,100.0
stock_movement_count,100.0



SLOW_MOVING_SUMMARY: 132 rows


,missing (%)
sale_line_id,100.0
transaction_id,100.0
sale_date,100.0
sale_month,100.0
shop_name,100.0
shop_type,100.0
county,100.0
town,100.0
neighbourhood,100.0
opening_date,100.0



STOCK_MOVEMENT: 14,837 rows


,missing (%)
transaction_id,100.0
sale_date,100.0
sale_month,100.0
town,100.0
shop_name,100.0
shop_type,100.0
county,100.0
opening_date,100.0
neighbourhood,100.0
shop_currency,100.0


In [15]:
print("Dataset shape:", df.shape)
print(df["record_type"].value_counts())

fields_to_check = {
    "SALE": ["sale_line_id", "sale_date", "quantity", "line_revenue_kes"],
    "PRODUCT_CATALOGUE": ["product_id", "product_name", "unit_cost_kes"],
    "STOCK_MOVEMENT": ["movement_id", "movement_type", "quantity_change_units"],
    "INVENTORY_SNAPSHOT": ["product_id", "quantity_on_hand", "stock_status"],
    "SLOW_MOVING_SUMMARY": ["product_id", "units_sold_last_90_days", "slow_moving"],
}

for record_type, columns in fields_to_check.items():
    group = df[df["record_type"] == record_type]
    print(f"\n{record_type}: {len(group)} rows")
    print("Non-empty relevant values:")
    print(group[columns].notna().sum())

Dataset shape: (28683, 55)
record_type
STOCK_MOVEMENT         14837
SALE                   13450
PRODUCT_CATALOGUE        132
INVENTORY_SNAPSHOT       132
SLOW_MOVING_SUMMARY      132
Name: count, dtype: int64

SALE: 13450 rows
Non-empty relevant values:
sale_line_id        13450
sale_date           13450
quantity            13450
line_revenue_kes    13450
dtype: int64

PRODUCT_CATALOGUE: 132 rows
Non-empty relevant values:
product_id       132
product_name     132
unit_cost_kes    132
dtype: int64

STOCK_MOVEMENT: 14837 rows
Non-empty relevant values:
movement_id              14837
movement_type            14837
quantity_change_units    14837
dtype: int64

INVENTORY_SNAPSHOT: 132 rows
Non-empty relevant values:
product_id          132
quantity_on_hand    132
stock_status        132
dtype: int64

SLOW_MOVING_SUMMARY: 132 rows
Non-empty relevant values:
product_id                 132
units_sold_last_90_days    132
slow_moving                132
dtype: int64


- The dataset contains **28,683 rows and 55 columns**. The row count is split across five record types.
- `notna().sum()` counts how many values are **present** in each selected column; it does not count missing values.
- Each record type uses different columns. For example, sale IDs are blank on inventory rows because those rows are not sales.
- The important check is that the fields relevant to each type are populated: sales have sale details, stock movements have movement details, and inventory snapshots have stock balances.
- The data is labelled **synthetic** and is for demonstrating DukaSense features, not reporting actual shop performance.

In [16]:
# Work on copies so the original df is unchanged.
sales = df.loc[df["record_type"] == "SALE"].copy()
movements = df.loc[df["record_type"] == "STOCK_MOVEMENT"].copy()
inventory = df.loc[df["record_type"] == "INVENTORY_SNAPSHOT"].copy()

issues = []

def check_issue(name, bad_rows):
    count = int(bad_rows.sum())
    if count:
        issues.append((name, count))
        print(f"CHECK: {name} — {count} problem row(s)")
    else:
        print(f"OK: {name}")

# Convert values in these copies for checking.
sales["sale_date"] = pd.to_datetime(sales["sale_date"], errors="coerce")
for col in [
    "quantity", "unit_selling_price_kes", "unit_cost_at_sale_kes",
    "line_revenue_kes", "line_estimated_cogs_kes",
    "line_estimated_gross_profit_kes",
]:
    sales[col] = pd.to_numeric(sales[col], errors="coerce")

movements["movement_datetime"] = pd.to_datetime(
    movements["movement_datetime"], errors="coerce"
)
for col in [
    "quantity_change_units", "movement_sequence",
    "balance_after_movement_units",
]:
    movements[col] = pd.to_numeric(movements[col], errors="coerce")

inventory["quantity_on_hand"] = pd.to_numeric(
    inventory["quantity_on_hand"], errors="coerce"
)
inventory["reorder_level"] = pd.to_numeric(
    inventory["reorder_level"], errors="coerce"
)

# Check sale IDs, dates, quantities, and financial calculations.
check_issue("Duplicate sale-line IDs", sales["sale_line_id"].duplicated())
check_issue("Invalid sale dates", sales["sale_date"].isna())
check_issue("Missing/invalid sales numbers", sales[
    ["quantity", "unit_selling_price_kes", "unit_cost_at_sale_kes",
     "line_revenue_kes", "line_estimated_cogs_kes",
     "line_estimated_gross_profit_kes"]
].isna().any(axis=1))
check_issue("Zero or negative sale quantities", sales["quantity"] <= 0)
check_issue("Negative prices or costs", (
    (sales["unit_selling_price_kes"] < 0)
    | (sales["unit_cost_at_sale_kes"] < 0)
))

check_issue("Revenue does not match quantity × selling price", ~np.isclose(
    sales["line_revenue_kes"],
    sales["quantity"] * sales["unit_selling_price_kes"],
    atol=0.01,
))
check_issue("COGS does not match quantity × unit cost", ~np.isclose(
    sales["line_estimated_cogs_kes"],
    sales["quantity"] * sales["unit_cost_at_sale_kes"],
    atol=0.01,
))
check_issue("Gross profit does not match revenue − COGS", ~np.isclose(
    sales["line_estimated_gross_profit_kes"],
    sales["line_revenue_kes"] - sales["line_estimated_cogs_kes"],
    atol=0.01,
))

# Check that each sale has exactly one matching stock-out movement.
sale_movements = movements.loc[
    movements["movement_type"] == "SALE_STOCK_OUT"
].copy()

check_issue("Duplicate sale IDs in stock-out movements",
            sale_movements["sale_line_id"].duplicated())

if not sales["sale_line_id"].duplicated().any() and not sale_movements[
    "sale_line_id"
].duplicated().any():
    sales_for_match = sales[
        ["sale_line_id", "shop_id", "product_id", "sale_date", "quantity"]
    ]
    movements_for_match = sale_movements[
        ["sale_line_id", "shop_id", "product_id",
         "movement_datetime", "quantity_change_units"]
    ]

    matched = sales_for_match.merge(
        movements_for_match,
        on="sale_line_id",
        how="outer",
        suffixes=("_sale", "_movement"),
        indicator=True,
        validate="one_to_one",
    )

    check_issue("Sales without a matching stock movement",
                matched["_merge"] != "both")

    matched_sales = matched.loc[matched["_merge"] == "both"]
    check_issue("Sale movement shop/product/date/quantity mismatch", (
        (matched_sales["shop_id_sale"] != matched_sales["shop_id_movement"])
        | (matched_sales["product_id_sale"] != matched_sales["product_id_movement"])
        | (matched_sales["sale_date"] != matched_sales["movement_datetime"])
        | ~np.isclose(
            matched_sales["quantity"],
            -matched_sales["quantity_change_units"],
            atol=0.001,
        )
    ))

# Calculate stock balances from the movement history.
movements = movements.sort_values(
    ["shop_id", "product_id", "movement_datetime", "movement_sequence"]
).copy()

movements["calculated_balance"] = movements.groupby(
    ["shop_id", "product_id"]
)["quantity_change_units"].cumsum()

check_issue("Negative stock balances", movements["calculated_balance"] < 0)
check_issue("Recorded movement balances disagree with ledger", ~np.isclose(
    movements["calculated_balance"],
    movements["balance_after_movement_units"],
    atol=0.001,
))

# Compare final ledger balances with the inventory snapshot.
ending_stock = (
    movements.groupby(["shop_id", "product_id"], as_index=False)
    .tail(1)[["shop_id", "product_id", "calculated_balance"]]
)

stock_check = inventory.merge(
    ending_stock,
    on=["shop_id", "product_id"],
    how="outer",
    indicator=True,
    validate="one_to_one",
)

check_issue("Snapshot products do not match ledger products",
            stock_check["_merge"] != "both")

matched_stock = stock_check.loc[stock_check["_merge"] == "both"]
check_issue("Snapshot quantity disagrees with final ledger balance", ~np.isclose(
    matched_stock["quantity_on_hand"],
    matched_stock["calculated_balance"],
    atol=0.001,
))

if issues:
    print(f"\nFound {len(issues)} type(s) of issue. No data was changed.")
else:
    print("\nAll checks passed. No data was changed.")

OK: Duplicate sale-line IDs
OK: Invalid sale dates
OK: Missing/invalid sales numbers
OK: Zero or negative sale quantities
OK: Negative prices or costs
OK: Revenue does not match quantity × selling price
OK: COGS does not match quantity × unit cost
OK: Gross profit does not match revenue − COGS
OK: Duplicate sale IDs in stock-out movements
OK: Sales without a matching stock movement
OK: Sale movement shop/product/date/quantity mismatch
OK: Negative stock balances
OK: Recorded movement balances disagree with ledger
OK: Snapshot products do not match ledger products
OK: Snapshot quantity disagrees with final ledger balance

All checks passed. No data was changed.


- The validation checks found no issues with the sales calculations, sale-to-stock movement links or inventory balances.
- No rows were dropped and no values were imputed.
- Blank fields on other record types are expected because the combined file contains different kinds of records.
- These checks cover the key sales and inventory relationships; they do not prove every possible data issue is absent.

In [17]:
sales_df = df.loc[df["record_type"] == "SALE"].copy()
products_df = df.loc[df["record_type"] == "PRODUCT_CATALOGUE"].copy()
stock_movements_df = df.loc[df["record_type"] == "STOCK_MOVEMENT"].copy()
inventory_df = df.loc[df["record_type"] == "INVENTORY_SNAPSHOT"].copy()
slow_moving_df = df.loc[df["record_type"] == "SLOW_MOVING_SUMMARY"].copy()

# Sales: dates and financial amounts
sales_df["sale_date"] = pd.to_datetime(sales_df["sale_date"], errors="raise")

sales_number_columns = [
    "quantity",
    "unit_selling_price_kes",
    "unit_cost_at_sale_kes",
    "line_revenue_kes",
    "line_estimated_cogs_kes",
    "line_estimated_gross_profit_kes",
]
sales_df[sales_number_columns] = sales_df[sales_number_columns].apply(
    pd.to_numeric, errors="raise"
)

# Products: prices, costs, and reorder levels
product_number_columns = [
    "unit_selling_price_kes",
    "unit_cost_kes",
    "reorder_level",
]
products_df[product_number_columns] = products_df[product_number_columns].apply(
    pd.to_numeric, errors="raise"
)

# Stock movements: timestamps and quantities
stock_movements_df["movement_datetime"] = pd.to_datetime(
    stock_movements_df["movement_datetime"], errors="raise"
)

movement_number_columns = [
    "quantity_change_units",
    "movement_sequence",
    "balance_after_movement_units",
]
stock_movements_df[movement_number_columns] = stock_movements_df[
    movement_number_columns
].apply(pd.to_numeric, errors="raise")

# Inventory snapshots: date and stock quantities
inventory_df["snapshot_date"] = pd.to_datetime(
    inventory_df["snapshot_date"], errors="raise"
)
inventory_df[["quantity_on_hand", "reorder_level"]] = inventory_df[
    ["quantity_on_hand", "reorder_level"]
].apply(pd.to_numeric, errors="raise")

# Slow-moving summary: blank last-sale dates are allowed for products with no sales
slow_moving_df["period_start"] = pd.to_datetime(
    slow_moving_df["period_start"], errors="raise"
)
slow_moving_df["period_end"] = pd.to_datetime(
    slow_moving_df["period_end"], errors="raise"
)
slow_moving_df["last_sale_date"] = pd.to_datetime(
    slow_moving_df["last_sale_date"], errors="coerce"
)
slow_moving_df["units_sold_last_90_days"] = pd.to_numeric(
    slow_moving_df["units_sold_last_90_days"], errors="raise"
)
slow_moving_df["threshold_units"] = pd.to_numeric(
    slow_moving_df["threshold_units"], errors="raise"
)

slow_flag = slow_moving_df["slow_moving"].astype("string").str.lower().str.strip()
slow_moving_df["slow_moving"] = slow_flag.map({
    "true": True,
    "false": False,
    "1": True,
    "0": False,
    "1.0": True,
    "0.0": False,
})

if slow_moving_df["slow_moving"].isna().any():
    raise ValueError("Unexpected value found in slow_moving.")

print("Date and numeric conversions completed.")
print("Sales date type:", sales_df["sale_date"].dtype)
print("Sales quantity type:", sales_df["quantity"].dtype)
print("Movement timestamp type:", stock_movements_df["movement_datetime"].dtype)
print("Inventory quantity type:", inventory_df["quantity_on_hand"].dtype)
print("Slow-moving flag type:", slow_moving_df["slow_moving"].dtype)

Date and numeric conversions completed.
Sales date type: datetime64[us]
Sales quantity type: float64
Movement timestamp type: datetime64[us]
Inventory quantity type: float64
Slow-moving flag type: bool


- The dataset contains 13,450 sales lines from 12 shops, covering January through December 2025.
- A sales line represents one product on a transaction; it is not the same as a transaction.
- The monthly counts show how many product lines were recorded, not the value of sales.
- December has the most sales lines (1,572). Because these are synthetic records, this should not be treated as evidence of actual seasonal demand.

In [18]:
print(f"Sales rows: {len(sales_df):,}")
print(f"Shops: {sales_df['shop_id'].nunique()}")
print(f"Products sold: {sales_df['product_id'].nunique()}")
print(f"Date range: {sales_df['sale_date'].min().date()} to "
      f"{sales_df['sale_date'].max().date()}")

print("\nSales lines per month:")
display(
    sales_df.groupby("sale_month")["sale_line_id"]
    .count()
    .rename("sales_lines")
    .to_frame()
)

Sales rows: 13,450
Shops: 12
Products sold: 120
Date range: 2025-01-01 to 2025-12-31

Sales lines per month:


,sales_lines
sale_month,
2025-01,1200
2025-02,961
2025-03,1042
2025-04,1095
2025-05,1220
2025-06,992
2025-07,1038
2025-08,1177
2025-09,1034


In [19]:
monthly_kpis = (
    sales_df.groupby("sale_month")
    .agg(
        revenue_kes=("line_revenue_kes", "sum"),
        cogs_kes=("line_estimated_cogs_kes", "sum"),
        gross_profit_kes=("line_estimated_gross_profit_kes", "sum"),
    )
)

monthly_kpis["gross_margin_percent"] = (
    monthly_kpis["gross_profit_kes"]
    .div(monthly_kpis["revenue_kes"].where(monthly_kpis["revenue_kes"] != 0))
    .mul(100)
)

display(monthly_kpis.round(2))

,revenue_kes,cogs_kes,gross_profit_kes,gross_margin_percent
sale_month,,,,
2025-01,312479.26,252969.76,59509.50,19.04
2025-02,246511.97,199377.99,47133.98,19.12
2025-03,280354.83,227741.91,52612.92,18.77
2025-04,281722.92,228039.31,53683.61,19.06
2025-05,320506.64,259684.25,60822.39,18.98
2025-06,253487.59,205472.43,48015.16,18.94
2025-07,260313.95,210555.71,49758.24,19.11
2025-08,292497.49,236564.32,55933.17,19.12
2025-09,282310.77,228904.72,53406.05,18.92


- Monthly revenue ranges from KES 246,511.97 in February to KES 409,982.00 in December.
- Gross margin is relatively steady in this dataset, at approximately 18.8%–19.1% each month.
- Gross profit is revenue minus cost of goods sold (COGS); it does not include expenses such as rent or wages, so it is not net profit.
- These are synthetic demo results.

In [20]:
product_performance = (
    sales_df.groupby(
        ["product_name", "product_category", "product_unit"],
        as_index=False
    )
    .agg(
        units_sold=("quantity", "sum"),
        revenue_kes=("line_revenue_kes", "sum"),
        gross_profit_kes=("line_estimated_gross_profit_kes", "sum"),
    )
)

print("Top 10 products by units sold:")
display(
    product_performance
    .sort_values("units_sold", ascending=False)
    .head(10)
    .round(2)
)

print("Top 10 products by revenue:")
display(
    product_performance
    .sort_values("revenue_kes", ascending=False)
    .head(10)
    .round(2)
)

Top 10 products by units sold:


,product_name,product_category,product_unit,units_sold,revenue_kes,gross_profit_kes
10,Mchele 1 kg,Staples,1 kg bag,3882.0,601365.90,117062.86
14,Soda 500 ml,Beverages,500 ml bottle,3815.0,228859.32,69098.67
5,Mafuta ya kupika 1 L,Cooking oil,1 L bottle,3744.0,916550.88,150567.16
9,Maziwa 500 ml,Dairy,500 ml packet,3722.0,223226.40,45116.88
15,Sukari 1 kg,Staples,1 kg bag,3717.0,631713.54,94176.23
16,Unga wa mahindi 2 kg,Staples,2 kg bag,3675.0,422317.95,84739.59
8,Mayai,Dairy,egg,3595.0,71883.32,18118.13
11,Mkate,Bakery,loaf,3518.0,228485.14,45985.94
1,Biskuti,Snacks,packet,585.0,14713.05,4261.89
7,Maji 1 L,Beverages,1 L bottle,376.0,16818.48,5200.08


Top 10 products by revenue:


,product_name,product_category,product_unit,units_sold,revenue_kes,gross_profit_kes
5,Mafuta ya kupika 1 L,Cooking oil,1 L bottle,3744.0,916550.88,150567.16
15,Sukari 1 kg,Staples,1 kg bag,3717.0,631713.54,94176.23
10,Mchele 1 kg,Staples,1 kg bag,3882.0,601365.90,117062.86
16,Unga wa mahindi 2 kg,Staples,2 kg bag,3675.0,422317.95,84739.59
14,Soda 500 ml,Beverages,500 ml bottle,3815.0,228859.32,69098.67
11,Mkate,Bakery,loaf,3518.0,228485.14,45985.94
9,Maziwa 500 ml,Dairy,500 ml packet,3722.0,223226.40,45116.88
8,Mayai,Dairy,egg,3595.0,71883.32,18118.13
17,Unga wa ngano 2 kg,Staples,2 kg bag,271.0,47117.35,7209.00
6,Maharagwe 1 kg,Pulses,1 kg bag,216.0,31697.58,7039.25


- Mchele 1 kg had the highest total quantity sold: 3,882 units across the shops.
- Mafuta ya kupika 1 L generated the highest revenue: KES 916,550.88.
- Quantity sold and revenue are different measures, so a product can lead one ranking without leading the other.
- These totals combine the synthetic sales from all shops. They do not establish actual customer demand.
- Products with zero sales are not included in this sales-based ranking.

In [22]:
inventory_view = inventory_df.merge(
    products_df[["shop_id", "product_id", "product_name"]].rename(
        columns={"product_name": "catalogue_product_name"}
    ),
    on=["shop_id", "product_id"],
    how="left",
    validate="one_to_one",
)

print("Stock status counts:")
display(inventory_view["stock_status"].value_counts().rename("products").to_frame())

print("Products marked for reorder on 2025-12-31:")
display(
    inventory_view.loc[
        inventory_view["stock_status"] == "REORDER",
        ["shop_id", "catalogue_product_name", "quantity_on_hand", "reorder_level"],
    ]
    .sort_values(["shop_id", "quantity_on_hand"])
    .reset_index(drop=True)
)

Stock status counts:


,products
stock_status,
OK,93
REORDER,39


Products marked for reorder on 2025-12-31:


,shop_id,catalogue_product_name,quantity_on_hand,reorder_level
0,SH001,Sukari 1 kg,1.0,7.0
1,SH001,Mkate,5.0,5.0
2,SH001,Mayai,5.0,18.0
3,SH001,Unga wa mahindi 2 kg,7.0,7.0
4,SH002,Sukari 1 kg,1.0,8.0
5,SH002,Soda 500 ml,3.0,12.0
6,SH002,Unga wa mahindi 2 kg,6.0,8.0
7,SH003,Maziwa 500 ml,5.0,9.0
8,SH003,Unga wa mahindi 2 kg,9.0,9.0
9,SH003,Mayai,10.0,22.0


- On 2025-12-31, 39 of 132 shop-product records were marked `REORDER`; the remaining 93 were marked `OK`.
- A product is marked `REORDER` when its stock is at or below its reorder level.
- These are synthetic demo records, so the counts demonstrate how stock alerts work; they do not describe real shops.

In [24]:
inventory_for_slow = inventory_df[
    ["shop_id", "product_id", "quantity_on_hand", "reorder_level", "stock_status"]
].rename(columns={
    "quantity_on_hand": "year_end_quantity",
    "reorder_level": "snapshot_reorder_level",
    "stock_status": "snapshot_stock_status",
})

slow_view = slow_moving_df.merge(
    inventory_for_slow,
    on=["shop_id", "product_id"],
    how="left",
    validate="one_to_one",
)

print("Slow-moving status:")
display(
    slow_view["slow_moving"]
    .value_counts()
    .rename(index={True: "Slow-moving", False: "Not slow-moving"})
    .rename("products")
    .to_frame()
)

print("Slow-moving products:")
display(
    slow_view.loc[
        slow_view["slow_moving"],
        [
            "shop_id",
            "product_name",
            "units_sold_last_90_days",
            "last_sale_date",
            "year_end_quantity",
            "snapshot_reorder_level",
            "snapshot_stock_status",
        ],
    ]
    .sort_values(["units_sold_last_90_days", "shop_id"])
    .reset_index(drop=True)
)

Slow-moving status:


,products
slow_moving,
Not slow-moving,117
Slow-moving,15


Slow-moving products:


,shop_id,product_name,units_sold_last_90_days,last_sale_date,year_end_quantity,snapshot_reorder_level,snapshot_stock_status
0,SH001,Sabuni ya kufulia 500 g,0.0,NaT,12.0,5.0,OK
1,SH002,Chumvi ya mezani 1 kg,0.0,NaT,12.0,8.0,OK
2,SH003,Juisi 500 ml,0.0,NaT,12.0,6.0,OK
3,SH004,Karanga 250 g,0.0,NaT,12.0,5.0,OK
4,SH005,Kiberiti,0.0,NaT,12.0,10.0,OK
5,SH006,Lotion 200 ml,0.0,NaT,12.0,3.0,OK
6,SH007,Maharagwe 2 kg,0.0,NaT,12.0,4.0,OK
7,SH008,Tissue paper,0.0,NaT,12.0,8.0,OK
8,SH009,Dishwashing liquid 500 ml,0.0,NaT,12.0,4.0,OK
9,SH010,Unga wa ngano 1 kg,0.0,NaT,12.0,6.0,OK


- The demo rule flags products with fewer than five units sold from October 3 to December 31, 2025.
- 15 of 132 shop-product pairs meet this rule: 12 had no sales in the period, and three sold fewer than five units.
- Three slow-moving products are also marked `REORDER`; slow sales and low stock are separate signals.
- This threshold is a demo assumption, not a validated business standard. The data are synthetic.

In [25]:
shop_performance = (
    sales_df.groupby(
        ["shop_id", "shop_name", "county"],
        as_index=False
    )
    .agg(
        transactions=("transaction_id", "nunique"),
        sales_lines=("sale_line_id", "count"),
        units_sold=("quantity", "sum"),
        revenue_kes=("line_revenue_kes", "sum"),
        cogs_kes=("line_estimated_cogs_kes", "sum"),
        gross_profit_kes=("line_estimated_gross_profit_kes", "sum"),
    )
)

shop_performance["gross_margin_percent"] = (
    shop_performance["gross_profit_kes"]
    .div(shop_performance["revenue_kes"].where(
        shop_performance["revenue_kes"] != 0
    ))
    .mul(100)
)

display(
    shop_performance
    .sort_values("revenue_kes", ascending=False)
    .round(2)
    .reset_index(drop=True)
)

,shop_id,shop_name,county,transactions,sales_lines,units_sold,revenue_kes,cogs_kes,gross_profit_kes,gross_margin_percent
0,SH010,Twende Duka,Machakos,627,1510,3596.0,382456.41,319673.89,62782.52,16.42
1,SH006,Pamoja Traders,Nakuru,576,1360,3272.0,359390.64,283797.75,75592.89,21.03
2,SH003,Upendo Duka,Nairobi,547,1267,3021.0,349056.85,282672.00,66384.85,19.02
3,SH005,Jirani Mart,Kiambu,506,1162,2728.0,316695.00,270743.74,45951.26,14.51
4,SH009,Imani Stores,Uasin Gishu,517,1175,2818.0,304410.18,255318.40,49091.78,16.13
5,SH012,Jitegemee Mini Mart,Kakamega,479,1101,2592.0,283202.51,222383.90,60818.61,21.48
6,SH002,Baraka Mini Shop,Nairobi,509,1165,2757.0,281436.19,226026.85,55409.34,19.69
7,SH008,Soko Fresh Shop,Mombasa,489,1100,2601.0,276504.07,220384.00,56120.07,20.30
8,SH011,Harambee Shop,Nyeri,462,975,2282.0,259703.93,203264.47,56439.46,21.73
9,SH004,Mwangaza Kiosk,Kiambu,433,966,2212.0,246809.01,199167.70,47641.31,19.30


- Twende Duka has the highest recorded revenue in this synthetic dataset: KES 382,456.41.
- Pamoja Traders has the highest gross profit: KES 75,592.89.
- Harambee Shop has the highest gross margin (21.73%); Jirani Mart has the lowest (14.51%).
- Revenue, gross profit, and gross margin measure different things and can rank shops differently.
- Gross profit excludes operating expenses, so it is not net profit.
- These are synthetic demo results, not evidence about real Kenyan shops.

In [26]:
# Total units sold for each shop-product pair
sales_by_product = (
    sales_df.groupby(["shop_id", "product_id"], as_index=False)
    .agg(units_sold=("quantity", "sum"))
)

# Combine sales history with the inventory snapshot
reorder_candidates = inventory_df[
    ["shop_id", "product_id", "quantity_on_hand", "reorder_level", "stock_status"]
].merge(
    sales_by_product,
    on=["shop_id", "product_id"],
    how="left"
)

reorder_candidates["units_sold"] = (
    reorder_candidates["units_sold"].fillna(0).astype(int)
)
reorder_candidates["units_below_reorder_level"] = (
    reorder_candidates["reorder_level"] - reorder_candidates["quantity_on_hand"]
).clip(lower=0)

reorder_candidates = (
    reorder_candidates[reorder_candidates["stock_status"] == "REORDER"]
    .sort_values(
        ["units_sold", "units_below_reorder_level"],
        ascending=[False, False]
    )
)

reorder_candidates.head(20)

,shop_id,product_id,quantity_on_hand,reorder_level,stock_status,units_sold,units_below_reorder_level
56,SH006,P-SH006-002,6.0,8.0,REORDER,421,2.0
106,SH010,P-SH010-008,2.0,11.0,REORDER,402,9.0
59,SH006,P-SH006-005,6.0,9.0,REORDER,397,3.0
60,SH006,P-SH006-006,5.0,7.0,REORDER,394,2.0
103,SH010,P-SH010-005,5.0,7.0,REORDER,381,2.0
28,SH003,P-SH003-007,10.0,22.0,REORDER,359,12.0
44,SH005,P-SH005-001,3.0,8.0,REORDER,357,5.0
61,SH006,P-SH006-007,11.0,22.0,REORDER,355,11.0
45,SH005,P-SH005-002,2.0,7.0,REORDER,353,5.0
26,SH003,P-SH003-005,5.0,9.0,REORDER,353,4.0


In [27]:
product_names = (
    products_df[["shop_id", "product_id", "product_name"]]
    .drop_duplicates()
)

reorder_candidates_named = reorder_candidates.merge(
    product_names,
    on=["shop_id", "product_id"],
    how="left"
)

reorder_candidates_named[
    [
        "shop_id",
        "product_name",
        "quantity_on_hand",
        "reorder_level",
        "units_sold",
        "units_below_reorder_level",
    ]
].head(20)

,shop_id,product_name,quantity_on_hand,reorder_level,units_sold,units_below_reorder_level
0,SH006,Mchele 1 kg,6.0,8.0,421,2.0
1,SH010,Soda 500 ml,2.0,11.0,402,9.0
2,SH006,Maziwa 500 ml,6.0,9.0,397,3.0
3,SH006,Mkate,5.0,7.0,394,2.0
4,SH010,Maziwa 500 ml,5.0,7.0,381,2.0
5,SH003,Mayai,10.0,22.0,359,12.0
6,SH005,Unga wa mahindi 2 kg,3.0,8.0,357,5.0
7,SH006,Mayai,11.0,22.0,355,11.0
8,SH005,Mchele 1 kg,2.0,7.0,353,5.0
9,SH003,Maziwa 500 ml,5.0,9.0,353,4.0


This list shows products marked `REORDER`, ranked by their total units sold during the year. 

For example, SH010 has only 2 Soda 500 ml in stock against a reorder level of 11, while SH006's Mchele 1 kg had the highest annual sales in this list (421 units).

`units_below_reorder_level` shows how far current stock is below the threshold. It does not account for supplier lead times, order sizes, or future demand, so it should not be treated as a final order recommendation. 

These results are based on synthetic demo data.

In [28]:
presentation_candidates = reorder_candidates_named.copy()

presentation_candidates["units_to_reach_reorder_level"] = (
    presentation_candidates["reorder_level"]
    - presentation_candidates["quantity_on_hand"]
).clip(lower=0).astype(int)

presentation_candidates = presentation_candidates[
    presentation_candidates["units_to_reach_reorder_level"] > 0
].sort_values(
    ["units_to_reach_reorder_level", "units_sold"],
    ascending=[False, False]
).head(10)

presentation_candidates["recommendation"] = (
    "Consider ordering at least "
    + presentation_candidates["units_to_reach_reorder_level"].astype(str)
    + " to reach the reorder level"
)

presentation_candidates[
    [
        "shop_id",
        "product_name",
        "quantity_on_hand",
        "reorder_level",
        "units_sold",
        "recommendation",
    ]
]

,shop_id,product_name,quantity_on_hand,reorder_level,units_sold,recommendation
27,SH007,Mayai,2.0,18.0,248,Consider ordering at least 16 to reach the reo...
30,SH001,Mayai,5.0,18.0,239,Consider ordering at least 13 to reach the reo...
5,SH003,Mayai,10.0,22.0,359,Consider ordering at least 12 to reach the reo...
7,SH006,Mayai,11.0,22.0,355,Consider ordering at least 11 to reach the reo...
1,SH010,Soda 500 ml,2.0,11.0,402,Consider ordering at least 9 to reach the reor...
17,SH002,Soda 500 ml,3.0,12.0,300,Consider ordering at least 9 to reach the reor...
10,SH009,Unga wa mahindi 2 kg,2.0,9.0,333,Consider ordering at least 7 to reach the reor...
12,SH004,Soda 500 ml,4.0,11.0,328,Consider ordering at least 7 to reach the reor...
14,SH005,Maziwa 500 ml,1.0,8.0,319,Consider ordering at least 7 to reach the reor...
18,SH002,Sukari 1 kg,1.0,8.0,295,Consider ordering at least 7 to reach the reor...


The table highlights products furthest below their reorder levels. 

SH007's Mayai has the largest gap: 16 units (2 on hand versus a reorder level of 18). 

SH010's Soda 500 ml is another item to review, with 2 on hand, a reorder level of 11, and 402 units sold over the year.

The suggested quantities only bring stock up to the reorder level. 

They do not account for supplier lead times, pack sizes, or future demand. 


In [33]:
sale = sales_df.iloc[0]

quantity = sale["quantity"]
selling_price = sale["unit_selling_price_kes"]
unit_cost = sale["unit_cost_at_sale_kes"]

revenue = quantity * selling_price
cogs = quantity * unit_cost
gross_profit = revenue - cogs
gross_margin = gross_profit / revenue * 100 if revenue else 0

print(f"Shop: {sale['shop_name']} ({sale['shop_id']})")
print(f"Product: {sale['product_name']}")
print(f"Quantity: {quantity}")
print(f"Calculated revenue: {quantity} × {selling_price} = KES {revenue:,.2f}")
print(f"Recorded revenue: KES {sale['line_revenue_kes']:,.2f}")
print(f"Calculated COGS: {quantity} × {unit_cost} = KES {cogs:,.2f}")
print(f"Recorded COGS: KES {sale['line_estimated_cogs_kes']:,.2f}")
print(f"Calculated gross profit: KES {gross_profit:,.2f}")
print(f"Recorded gross profit: KES {sale['line_estimated_gross_profit_kes']:,.2f}")
print(f"Calculated gross margin: {gross_margin:.2f}%")

Shop: Tumaini General Store (SH001)
Product: Mafuta ya kupika 1 L
Quantity: 3.0
Calculated revenue: 3.0 × 240.59 = KES 721.77
Recorded revenue: KES 721.77
Calculated COGS: 3.0 × 198.85 = KES 596.55
Recorded COGS: KES 596.55
Calculated gross profit: KES 125.22
Recorded gross profit: KES 125.22
Calculated gross margin: 17.35%


For one sale of 3 units of Mafuta ya kupika 1 L, calculated revenue (KES 721.77), COGS (KES 596.55), and gross profit (KES 125.22) matched the values recorded in the dataset. 

The calculated gross margin was 17.35%. 

This confirms the calculations for this row; it does not by itself validate every sale.